# CLaMP 3 GPU score extraction

Run the existing `encoding-music-embeddings` pipeline from a locally built wheel on a Google Colab CUDA runtime. Setup is the only network-enabled model operation; extraction is explicitly offline. The final SQLite database, pipeline manifest, and checksums are packaged for download to your local machine.

Before connecting, run `uv build --wheel` locally. Connect this notebook to a Colab GPU, then upload the single resulting `dist/encoding_music_mcp-*.whl` into `/content` with the VS Code Colab extension. The complete MEI corpus bundled in that wheel is the default input; optional additional inputs must be absolute paths in the Colab runtime.

> **Important:** Colab storage is temporary. A successful run is not complete until you download the archive in the final cell through the VS Code Colab extension's remote **Contents** view (or enable the browser download option).

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

wheel_upload = widgets.FileUpload(accept=".whl", multiple=False)
display(wheel_upload)

In [ ]:
# Configuration: edit these values before running the notebook.
from pathlib import Path

WHEEL_UPLOAD_DIR = Path("/content")
ADDITIONAL_MEI_INPUTS = []  # Optional absolute Colab file/directory paths.

RUNTIME_ROOT = Path("/content/encoding-music-work")
PROJECT_ENV_DIR = RUNTIME_ROOT / "project-env"
CLAMP_ENV_DIR = RUNTIME_ROOT / "clamp3-env"
OUTPUT_DIR = RUNTIME_ROOT / "runs"
DATABASE_PATH = RUNTIME_ROOT / "score-embeddings.sqlite3"
ARTIFACT_BASENAME = "encoding-music-embeddings"
CLAMP_TIMEOUT_SECONDS = 7200
MINIMUM_VRAM_GB = 5.0
CUDA_WHEEL_INDEX = "https://download.pytorch.org/whl/cu128"
TORCH_PACKAGES = ["torch", "torchvision", "torchaudio"]

# Drive is optional and may persist only the reusable CLaMP cache.
USE_GOOGLE_DRIVE_CACHE = False
GOOGLE_DRIVE_CACHE_DIR = ""  # Example: /content/drive/MyDrive/clamp3-cache
BROWSER_DOWNLOAD = False  # Keep False when using the VS Code Colab extension.

In [ ]:
# Fail early if this notebook kernel cannot see a suitable CUDA GPU.
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable in the notebook kernel. In Colab select "
        "Runtime > Change runtime type > GPU, reconnect, and rerun."
    )
gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / (1024**3)
if vram_gb < MINIMUM_VRAM_GB:
    raise RuntimeError(
        f"GPU {gpu.name!r} has {vram_gb:.1f} GiB VRAM; "
        f"at least {MINIMUM_VRAM_GB:.1f} GiB is required. Select a larger GPU runtime."
    )
print(f"Notebook GPU: {gpu.name} ({vram_gb:.1f} GiB VRAM), CUDA {torch.version.cuda}")

In [ ]:
# Install exactly one locally built project wheel into an isolated environment.
import shutil
import subprocess
import sys
import zipfile

def run(command, *, cwd=None, env=None, check=True):
    print("+", " ".join(map(str, command)))
    return subprocess.run(
        [str(part) for part in command], cwd=cwd, env=env, check=check, text=True
    )

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
if shutil.which("uv") is None:
    run([sys.executable, "-m", "pip", "install", "uv"])
uploaded_wheels = sorted(WHEEL_UPLOAD_DIR.glob("encoding_music_mcp-*.whl"))
if not uploaded_wheels:
    raise FileNotFoundError(
        "No uploaded encoding_music_mcp-*.whl was found in /content. "
        "Locally run `uv build --wheel`, then upload the wheel from dist/ "
        "with the VS Code Colab extension and rerun this cell."
    )
if len(uploaded_wheels) > 1:
    raise RuntimeError(
        "Multiple encoding_music_mcp-*.whl files were found in /content: "
        f"{[path.name for path in uploaded_wheels]}. Keep exactly one and rerun."
    )
PROJECT_WHEEL = uploaded_wheels[0]
with zipfile.ZipFile(PROJECT_WHEEL) as wheel_archive:
    wheel_members = set(wheel_archive.namelist())
if "encoding_music_mcp/score_embeddings/pipeline.py" not in wheel_members:
    raise RuntimeError(
        f"{PROJECT_WHEEL.name} does not contain the score-embedding pipeline. "
        "Rebuild the wheel from the updated local project."
    )
bundled_mei_members = [
    name for name in wheel_members
    if name.startswith("encoding_music_mcp/resources/mei_files/")
    and name.endswith(".mei")
]
if not bundled_mei_members:
    raise RuntimeError(
        f"{PROJECT_WHEEL.name} contains no bundled MEI resources. "
        "Check the wheel build configuration and rebuild it."
    )
if not (PROJECT_ENV_DIR / "bin/python").exists():
    run(["uv", "venv", "--python", "3.12", PROJECT_ENV_DIR])
PROJECT_PYTHON = PROJECT_ENV_DIR / "bin/python"
run(["uv", "pip", "install", "--python", PROJECT_PYTHON,
     f"{PROJECT_WHEEL}[score-embeddings]"])
PIPELINE_CLI = PROJECT_ENV_DIR / "bin/encoding-music-embeddings"
if not PIPELINE_CLI.is_file():
    raise RuntimeError("The installed wheel did not create encoding-music-embeddings")
resource_probe = subprocess.run(
    [str(PROJECT_PYTHON), "-c",
     'from importlib.resources import files; print(files("encoding_music_mcp.resources").joinpath("mei_files"))'],
    check=True, capture_output=True, text=True,
)
BUNDLED_MEI_DIRECTORY = Path(resource_probe.stdout.strip())
if not BUNDLED_MEI_DIRECTORY.is_dir():
    raise RuntimeError(f"Installed bundled MEI directory is missing: {BUNDLED_MEI_DIRECTORY}")
print(f"Installed {PROJECT_WHEEL.name}; bundled corpus: {BUNDLED_MEI_DIRECTORY}")

In [ ]:
# Provision the separate ADR-0001 Python 3.10/PyTorch CLaMP runtime.
CLAMP_REQUIREMENTS = [
    "nnAudio==0.3.3", "transformers==4.40.0", "accelerate==0.34.0",
    "scikit-learn==1.5.1", "numpy==1.26.4", "tqdm==4.66.5",
    "unidecode==1.3.6", "wandb==0.17.8", "pillow==9.5.0",
    "mido==1.3.0", "samplings==0.1.7", "abctoolkit==0.0.4",
    "soundfile==0.12.1",
]
if not (CLAMP_ENV_DIR / "bin/python").exists():
    run(["uv", "venv", "--python", "3.10", CLAMP_ENV_DIR])
CLAMP_PYTHON = CLAMP_ENV_DIR / "bin/python"
run(["uv", "pip", "install", "--python", CLAMP_PYTHON,
     "--index-url", CUDA_WHEEL_INDEX, *TORCH_PACKAGES])
run(["uv", "pip", "install", "--python", CLAMP_PYTHON, *CLAMP_REQUIREMENTS])

In [ ]:
# Verify CUDA again inside the separate interpreter used by CLaMP.
import json

probe_code = """
import json, torch
available = torch.cuda.is_available()
payload = {"available": available, "torch": torch.__version__, "cuda": torch.version.cuda}
if available:
    props = torch.cuda.get_device_properties(0)
    payload.update(name=props.name, vram_gb=props.total_memory / (1024**3))
print(json.dumps(payload))
"""
probe = subprocess.run(
    [str(CLAMP_PYTHON), "-c", probe_code], check=True, capture_output=True, text=True
)
clamp_gpu = json.loads(probe.stdout.strip())
if not clamp_gpu["available"]:
    raise RuntimeError(
        "The separate CLaMP Python reports torch.cuda.is_available() == False. "
        "Check CUDA_WHEEL_INDEX/TORCH_PACKAGES, reinstall, and rerun this probe."
    )
if clamp_gpu["vram_gb"] < MINIMUM_VRAM_GB:
    raise RuntimeError(
        f"CLaMP sees only {clamp_gpu['vram_gb']:.1f} GiB VRAM; "
        f"at least {MINIMUM_VRAM_GB:.1f} GiB is required."
    )
print("CLaMP GPU probe:", clamp_gpu)

In [ ]:
# Optional Drive persistence is deliberately restricted to model cache data.
if USE_GOOGLE_DRIVE_CACHE:
    if not GOOGLE_DRIVE_CACHE_DIR:
        raise ValueError("Set GOOGLE_DRIVE_CACHE_DIR when Drive cache is enabled")
    from google.colab import drive
    drive.mount("/content/drive")
    CLAMP_CACHE_DIR = Path(GOOGLE_DRIVE_CACHE_DIR)
else:
    CLAMP_CACHE_DIR = RUNTIME_ROOT / "clamp3-cache"
CLAMP_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Explicit online setup: pinned source, weights, dependencies, and readiness manifest.
run([PIPELINE_CLI, "setup",
     "--clamp-python", CLAMP_PYTHON,
     "--cache-dir", CLAMP_CACHE_DIR,
     "--timeout", str(CLAMP_TIMEOUT_SECONDS)])

In [ ]:
# Invoke the existing pipeline interface; do not recreate conversion or storage logic here.
import os

additional_inputs = [Path(value) for value in ADDITIONAL_MEI_INPUTS]
relative_inputs = [str(path) for path in additional_inputs if not path.is_absolute()]
if relative_inputs:
    raise ValueError(
        "Additional MEI inputs must be absolute Colab paths: "
        f"{relative_inputs}"
    )
MEI_INPUTS = [BUNDLED_MEI_DIRECTORY]
MEI_INPUTS.extend(additional_inputs)
resolved_inputs = MEI_INPUTS
missing_inputs = [str(path) for path in resolved_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"Configured MEI inputs do not exist: {missing_inputs}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
before_manifests = set(OUTPUT_DIR.glob("*/manifest.json"))
offline_env = os.environ.copy()
offline_env.update({
    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1", "WANDB_MODE": "offline",
})
extraction = run([
    PIPELINE_CLI, "extract", *resolved_inputs,
    "--database", DATABASE_PATH,
    "--output-dir", OUTPUT_DIR,
    "--clamp-python", CLAMP_PYTHON,
    "--cache-dir", CLAMP_CACHE_DIR,
    "--timeout", str(CLAMP_TIMEOUT_SECONDS),
], env=offline_env, check=False)
new_manifests = set(OUTPUT_DIR.glob("*/manifest.json")) - before_manifests
if len(new_manifests) != 1:
    raise RuntimeError(
        f"Extraction produced {len(new_manifests)} new manifests; expected one "
        f"(CLI exit code {extraction.returncode}). Inspect the preceding diagnostics."
    )
PIPELINE_MANIFEST = new_manifests.pop()
manifest_data = json.loads(PIPELINE_MANIFEST.read_text(encoding="utf-8"))
if manifest_data.get("run_error"):
    raise RuntimeError(f"Extraction failed: {manifest_data['run_error']}")
if not DATABASE_PATH.is_file():
    raise RuntimeError("Extraction completed without creating the SQLite database")
embedded = sum(item.get("embedding_id") is not None for item in manifest_data["scores"])
print(f"Run {manifest_data['run_id']}: {embedded}/{len(manifest_data['scores'])} embeddings stored")

In [ ]:
# Open the database through the public repository API, then build a portable archive.
import hashlib
import tarfile
import tempfile

verify_code = """
import sys
from encoding_music_mcp.score_embeddings.storage import EmbeddingRepository
with EmbeddingRepository(sys.argv[1]) as repository:
    count = repository.connection.execute("SELECT COUNT(*) FROM score_embeddings").fetchone()[0]
print(count)
"""
verification = subprocess.run(
    [str(PROJECT_PYTHON), "-c", verify_code, str(DATABASE_PATH)],
    check=True, capture_output=True, text=True,
)
print(f"EmbeddingRepository opened the database ({verification.stdout.strip()} rows).")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

ARTIFACT_ARCHIVE = RUNTIME_ROOT / f"{ARTIFACT_BASENAME}-{manifest_data['run_id']}.tar.gz"
with tempfile.TemporaryDirectory(prefix="embedding-artifact-") as temporary:
    payload = Path(temporary)
    packaged_database = payload / DATABASE_PATH.name
    packaged_manifest = payload / "manifest.json"
    shutil.copy2(DATABASE_PATH, packaged_database)
    shutil.copy2(PIPELINE_MANIFEST, packaged_manifest)
    checksums = payload / "SHA256SUMS"
    checksums.write_text(
        f"{sha256_file(packaged_database)}  {packaged_database.name}\n"
        f"{sha256_file(packaged_manifest)}  {packaged_manifest.name}\n",
        encoding="utf-8",
    )
    with tarfile.open(ARTIFACT_ARCHIVE, "w:gz") as archive:
        for item in (packaged_database, packaged_manifest, checksums):
            archive.add(item, arcname=item.name)
print(f"Archive: {ARTIFACT_ARCHIVE} ({ARTIFACT_ARCHIVE.stat().st_size / 1024**2:.1f} MiB)")

## Required: download the local artifact

The archive is still on an ephemeral Colab VM. Run the final cell, then use the VS Code Colab extension's remote **Contents** view to download the displayed `.tar.gz` file into your local checkout. Do not disconnect until the local copy is verified.

In [ ]:
banner = (
    "\n" + "=" * 78 + "\nDOWNLOAD REQUIRED BEFORE DISCONNECTING\n"
    f"Download this file from the remote Contents view:\n{ARTIFACT_ARCHIVE}\n"
    + "=" * 78
)
print(banner)
if BROWSER_DOWNLOAD:
    from google.colab import files
    files.download(str(ARTIFACT_ARCHIVE))
else:
    print("VS Code: right-click the archive in the Colab remote Contents view and download it.")